# Larp-LLM · Chat
Colab / Kaggle • Unsloth Gemma 3 12B adapter • Gradio 6

Choose a **single NVIDIA GPU** runtime (16 GB may be tight; 24 GB+ recommended), enable Internet on Kaggle, then run cells in order. Installation and first model download take time. This is text-only inference.

After installing, **restart the session/kernel once**, then continue from configuration. Do not run the install cell again after restarting. The launch cell creates a temporary public link by default; anyone with it can use your GPU. Set `SHARE=False` for local access.


## 1 · Install dependencies
Use a fresh Colab/Kaggle runtime. Keep the platform’s Torch/CUDA stack; do not force unrelated storage-library versions. Unsloth dependencies vary with the GPU/runtime, so its resolver selects the compatible stack. Exact resolved versions are printed below.

In [ ]:
%pip install --upgrade "unsloth" "gradio>=6,<7" "peft>=0.19.1"
%pip check


## 2 · Configuration and runtime checks
Run after restarting. Optional Hugging Face credentials: use Colab Secret `HF_TOKEN`, Kaggle Secret `HF_TOKEN`, or the environment variable. Never paste tokens into saved cells. Accept any required base-model license on Hugging Face if access is denied.

In [ ]:
import os
from importlib.metadata import version
MODEL_ID = "celestial-sora/Larp-LLM"
MAX_SEQ_LENGTH = 2048
SHARE = True
DEFAULT_SYSTEM = "You are Larper LLM. Respond in an internet-native, playful, meme-aware style. Be concise, lean into larping and know-ball humor, and match the user's language."
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except userdata.SecretNotFoundError:
        pass
if not HF_TOKEN and "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No accessible Kaggle HF_TOKEN secret; using public access.")
# Import Unsloth before Torch/Transformers so its patches are applied first.
from unsloth import FastModel
import torch
import gradio as gr
if not torch.cuda.is_available():
    raise RuntimeError("Enable a NVIDIA GPU runtime before loading the model.")
for package in ("unsloth", "unsloth_zoo", "torch", "transformers", "peft", "gradio"):
    print(f"{package}: {version(package)}")
if int(version("gradio").split(".")[0]) != 6:
    raise RuntimeError("Install Gradio 6 and restart the kernel.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 3 · Load adapter and its base model
Unsloth reads the adapter configuration and loads its Gemma 3 12B 4-bit base. Reduce context length if memory is tight. This cell downloads several GB; it does not train the model.

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=HF_TOKEN,
)
FastModel.for_inference(model)
model.eval()
print("Loaded:", MODEL_ID)


## 4 · Generation and measured statistics
`inputs` lives inside the generation callback. Timing synchronizes CUDA before and after `model.generate`; it includes prompt prefill and decoding, excludes tokenization and display. Token count is the actual output ID count, including an emitted EOS/end-of-turn token. First-run compilation can lower throughput. Long conversations produce a clear context-limit error instead of silently discarding messages.

In [ ]:
import time
from threading import Lock
GENERATION_LOCK = Lock()

def text_content(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content
                       if isinstance(block, dict) and block.get("type") == "text")
    return ""

def build_messages(message, history, system_prompt):
    messages = []
    if system_prompt.strip():
        messages.append({"role": "system", "content": [{"type": "text", "text": system_prompt.strip()}]})
    for item in history or []:
        if isinstance(item, dict) and item.get("role") in ("user", "assistant"):
            text = text_content(item.get("content"))
            if text.strip():
                messages.append({"role": item["role"], "content": [{"type": "text", "text": text}]})
    messages.append({"role": "user", "content": [{"type": "text", "text": message}]})
    return messages

def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)

def respond(message, history, system_prompt, temperature, top_p, top_k,
            min_p, max_new_tokens, repetition_penalty, seed):
    if not message.strip():
        raise gr.Error("Enter a message first.")
    # A shared model must not generate concurrently or have its RNG reseeded mid-run.
    with GENERATION_LOCK:
        inputs = tokenizer.apply_chat_template(
            build_messages(message, history, system_prompt), tokenize=True,
            add_generation_prompt=True, return_tensors="pt", return_dict=True,
        )
        device = model.get_input_embeddings().weight.device
        inputs = {key: value.to(device) for key, value in inputs.items()}
        prompt_tokens = inputs["input_ids"].shape[-1]
        if prompt_tokens + int(max_new_tokens) > MAX_SEQ_LENGTH:
            raise gr.Error("Context limit reached. Start a new chat, shorten the system prompt, or lower Max new tokens.")
        kwargs = dict(max_new_tokens=int(max_new_tokens), do_sample=float(temperature) > 0,
                      repetition_penalty=float(repetition_penalty), use_cache=True)
        if kwargs["do_sample"]:
            kwargs.update(temperature=float(temperature), top_p=float(top_p),
                          top_k=int(top_k), min_p=float(min_p))
        # Processor and tokenizer objects expose special IDs differently.
        text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
        pad_id = text_tokenizer.pad_token_id
        if pad_id is None:
            pad_id = text_tokenizer.eos_token_id
        if pad_id is not None:
            kwargs["pad_token_id"] = pad_id
        if int(seed) >= 0:
            torch.manual_seed(int(seed))
        try:
            synchronize(device)
            started = time.perf_counter()
            with torch.inference_mode():
                output = model.generate(**inputs, **kwargs)
            synchronize(device)
            elapsed = time.perf_counter() - started
        except torch.cuda.OutOfMemoryError as exc:
            raise gr.Error("GPU memory exhausted. Lower context/output length or choose a larger GPU.") from exc
        generated = output[0, prompt_tokens:]
        count = int(generated.numel())
        answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
        stats = f"**{count:,} tok** · **{count / max(elapsed, 1e-9):.2f} tok/s** · **{elapsed:.2f} s** · {prompt_tokens:,} input tok"
        return answer or "(No visible text generated.)", stats


## 5 · Build the chat UI
Chat first, with collapsed advanced controls. Temperature 0 uses greedy decoding. Seed −1 leaves the RNG unseeded; fixed seeds are best-effort reproducibility, not a guarantee across hardware. Statistics describe the last completed response. Clear starts a new chat.

In [ ]:
CSS = """
.gradio-container {max-width: 1080px !important; margin: auto; padding: 20px !important;}
#larp-shell {min-width: 0; border-radius: 22px;}
#larp-chat {height: clamp(300px, 56dvh, 680px) !important; overflow-wrap: anywhere;}
textarea, input {font-size: 16px !important;}
button {min-height: 44px;}
@media (max-width: 768px) {
 .gradio-container {padding: 10px !important;}
 #larp-chat {height: 52dvh !important; min-height: 280px;}
}
@media (max-width: 480px) {
 .gradio-container {padding: 8px !important; padding-bottom: max(8px, env(safe-area-inset-bottom)) !important;}
 #larp-chat {height: 48dvh !important;}
}
"""
with gr.Blocks(title="Larp-LLM") as demo:
    with gr.Column(elem_id="larp-shell"):
        gr.Markdown("# Larp-LLM\nYour chat, your settings. Gemma 3 · Unsloth")
        chatbot = gr.Chatbot(elem_id="larp-chat", height=520, show_label=False,
                             placeholder="Say something. พิมพ์ภาษาไทยได้เลย")
        stats = gr.Markdown("Send a message to measure tokens and speed.")
        with gr.Accordion("Advanced generation settings", open=False):
            system_prompt = gr.Textbox(label="System prompt", value=DEFAULT_SYSTEM, lines=3)
            with gr.Row():
                temperature = gr.Slider(0, 1.5, value=0.8, step=0.05, label="Temperature")
                top_p = gr.Slider(0.05, 1, value=0.9, step=0.01, label="Top-p")
                top_k = gr.Slider(0, 128, value=64, step=1, label="Top-k (0 disables)")
            with gr.Row():
                min_p = gr.Slider(0, 0.5, value=0, step=0.01, label="Min-p")
                max_new_tokens = gr.Slider(16, 1024, value=256, step=16, label="Max new tokens")
                repetition_penalty = gr.Slider(1, 1.5, value=1.05, step=0.01, label="Repetition penalty")
            seed = gr.Number(value=-1, minimum=-1, maximum=2147483647, precision=0, label="Seed (-1 = random)")
        chat = gr.ChatInterface(
            fn=respond, chatbot=chatbot,
            textbox=gr.Textbox(placeholder="Message Larp-LLM…", lines=2, max_lines=6, autofocus=False),
            additional_inputs=[system_prompt, temperature, top_p, top_k, min_p,
                               max_new_tokens, repetition_penalty, seed],
            additional_outputs=[stats],
            examples=[["bro think he know ball", DEFAULT_SYSTEM, 0.8, 0.9, 64, 0, 256, 1.05, -1],
                      ["อธิบาย aura economics ให้ฟังหน่อย", DEFAULT_SYSTEM, 0.8, 0.9, 64, 0, 256, 1.05, -1]],
            cache_examples=False, concurrency_limit=1,
        )
        chatbot.clear(lambda: "New chat. Send a message to measure tokens and speed.", outputs=stats, queue=False)


## 6 · Launch
Open the share URL in a full browser tab on desktop/iPad/mobile for more space. Keep the runtime active. Interrupt this cell and run the shutdown cell to stop serving. Responses appear when generation completes; measurements are not estimates from text length.

In [ ]:
demo.queue(max_size=16, default_concurrency_limit=1)
demo.launch(share=SHARE, debug=True, theme=gr.themes.Soft(), css=CSS)


## 7 · Shutdown (optional)
Run when finished; closes the app server. Disconnect the runtime to release GPU allocation.

In [ ]:
demo.close()


## Troubleshooting and references
- No GPU: enable GPU and rerun configuration onward.
- Import/dependency errors after install: restart once; inspect `%pip check`. Prefer a fresh runtime rather than repeatedly upgrading an already imported Torch stack.
- Access denied: accept the Gemma license if requested and configure `HF_TOKEN` via secrets.
- Out of memory: shorter context/output, one GPU with more memory, or restart to remove stale models. Two Kaggle T4s are not a single 32 GB GPU.
- Metrics include prefill, decoding and first-run overhead; they are not pure decode-only benchmarks.
- Gradio history accepts text strings and Gradio 6 text blocks. Files are not sent to the model in this text-only app.

[Gradio 6 migration](https://www.gradio.app/guides/gradio-6-migration-guide) · [Unsloth Gemma 3](https://docs.unsloth.ai/basics/tutorial-how-to-run-and-fine-tune-gemma-3) · [Adapter configuration](https://huggingface.co/celestial-sora/Larp-LLM/blob/main/adapter_config.json)
